In [1]:
from pathlib import Path
import pandas as pd

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

eclipse_dir = project_root / "data" / "raw" / "eclipse"

if not eclipse_dir.exists():
    print("Directory not found:", eclipse_dir)
else:
    for file in sorted(eclipse_dir.rglob("*")):
        if file.is_file():
            print(
                file.relative_to(eclipse_dir),
                f"({file.stat().st_size / 1024 / 1024:.2f} MB)"
            )

eclipse_platform.csv (105.78 MB)


In [2]:
eclipse_path = eclipse_dir / "eclipse_platform.csv"

df_eclipse = pd.read_csv(eclipse_path, low_memory=False)

print("Shape:", df_eclipse.shape)

print("\nColumns:")
print(df_eclipse.columns.tolist())

print("\nFirst 5 rows:")
display(df_eclipse.head(5))

Shape: (85156, 11)

Columns:
['Issue_id', 'Priority', 'Component', 'Duplicated_issue', 'Title', 'Description', 'Status', 'Resolution', 'Version', 'Created_time', 'Resolved_time']

First 5 rows:


,Issue_id,Priority,Component,Duplicated_issue,Title,Description,Status,Resolution,Version,Created_time,Resolved_time
0,1,P3,Team,NaN,Usability issue with external editors (1GE6IRL),- Setup a project that contains a *.gif resour...,CLOSED,FIXED,2.0,2001-10-10 21:34:00 -0400,2012-02-09 15:57:47 -0500
1,2,P5,Team,NaN,Opening repository resources doesnt honor type...,Opening repository resource always open the de...,RESOLVED,FIXED,2.0,2001-10-10 21:34:00 -0400,2002-05-07 10:33:56 -0400
2,3,P5,Team,NaN,Sync does not indicate deletion (1GIEN83),KM (10/2/2001 5:55:18 PM); \tThis PR about the...,RESOLVED,FIXED,2.0,2001-10-10 21:34:00 -0400,2010-05-07 10:28:53 -0400
3,4,P5,Team,NaN,need better error message if catching up over ...,- become synchronized with some project in a r...,RESOLVED,FIXED,2.0,2001-10-10 21:34:00 -0400,2002-03-01 16:27:31 -0500
4,5,P3,Team,NaN,ISharingManager sharing API inconsistent (1GAU...,For getting/setting the managed state of a res...,RESOLVED,WONTFIX,2.0,2001-10-10 21:34:00 -0400,2008-08-15 08:04:36 -0400


In [3]:
print("Missing values:")
print(df_eclipse.isna().sum())

print("\nData types:")
print(df_eclipse.dtypes)

print("\nDuplicate rows:")
print(df_eclipse.duplicated().sum())

Missing values:
Issue_id                0
Priority                0
Component               0
Duplicated_issue    70752
Title                   0
Description           129
Status                  0
Resolution              0
Version                 0
Created_time            0
Resolved_time           0
dtype: int64

Data types:
Issue_id              int64
Priority                str
Component               str
Duplicated_issue    float64
Title                   str
Description             str
Status                  str
Resolution              str
Version                 str
Created_time            str
Resolved_time           str
dtype: object

Duplicate rows:
0


In [4]:
print("Total reports:", len(df_eclipse))

print(
    "Unique issue IDs:",
    df_eclipse["Issue_id"].nunique()
)

print(
    "Reports with duplicate references:",
    df_eclipse["Duplicated_issue"].notna().sum()
)

print(
    "Reports without duplicate references:",
    df_eclipse["Duplicated_issue"].isna().sum()
)

display(
    df_eclipse.loc[
        df_eclipse["Duplicated_issue"].notna(),
        [
            "Issue_id",
            "Duplicated_issue",
            "Title",
            "Resolution"
        ]
    ].head(10)
)

Total reports: 85156
Unique issue IDs: 85156
Reports with duplicate references: 14404
Reports without duplicate references: 70752


,Issue_id,Duplicated_issue,Title,Resolution
24,25,28126.0,[CVS UI] Need VCM Prefs - Default Repo Connect...,DUPLICATE
39,40,20.0,Need connect to team stream (1GCQPKW),DUPLICATE
47,48,22.0,Make sure can future store other project refer...,DUPLICATE
60,61,60.0,.vcm_meta showing as change (1GDQTGW),DUPLICATE
93,94,2.0,Repositories View: all file types open to the ...,DUPLICATE
97,98,26.0,Need,DUPLICATE
121,122,170.0,CVS communication error (1GENYTQ),DUPLICATE
145,146,170.0,UnknownHostException: Repositories (1GEWRWG),DUPLICATE
159,160,22828.0,Option for CVS Server Name,DUPLICATE
170,171,4904.0,missing repository can leave user with no reso...,DUPLICATE


In [5]:
issue_ids = set(df_eclipse["Issue_id"])

duplicate_refs = df_eclipse[
    "Duplicated_issue"
].dropna().astype("int64")


valid_refs = duplicate_refs.isin(issue_ids)

print("Total duplicate references:", len(duplicate_refs))

print("References found in dataset:", valid_refs.sum())

print("References missing from dataset:", (~valid_refs).sum())


self_refs = df_eclipse[
    df_eclipse["Duplicated_issue"].notna()
    & (
        df_eclipse["Issue_id"]
        == df_eclipse["Duplicated_issue"]
    )
]

print("Self-referencing issues:", len(self_refs))

Total duplicate references: 14404
References found in dataset: 13231
References missing from dataset: 1173
Self-referencing issues: 0


In [6]:
df_eclipse_clean = df_eclipse.copy()


df_eclipse_clean["Created_time"] = pd.to_datetime(
    df_eclipse_clean["Created_time"],
    utc=True,
    errors="coerce"
)

print("Earliest issue:")
print(df_eclipse_clean["Created_time"].min())

print("\nLatest issue:")
print(df_eclipse_clean["Created_time"].max())

print("\nInvalid creation timestamps:")
print(df_eclipse_clean["Created_time"].isna().sum())

Earliest issue:
2001-10-11 01:34:00+00:00

Latest issue:
2013-12-30 23:05:00+00:00

Invalid creation timestamps:
0


In [7]:
df_eclipse_clean["Title"] = (
    df_eclipse_clean["Title"].fillna("").astype(str)
)

df_eclipse_clean["Description"] = (
    df_eclipse_clean["Description"].fillna("").astype(str)
)


df_eclipse_clean["issue_text"] = (
    df_eclipse_clean["Title"].str.strip()
    + "\n"
    + df_eclipse_clean["Description"].str.strip()
).str.strip()

print("Original shape:", df_eclipse.shape)
print("Cleaned shape:", df_eclipse_clean.shape)

print(
    "Empty issue texts:",
    df_eclipse_clean["issue_text"].eq("").sum()
)

print(
    "Missing descriptions after cleaning:",
    df_eclipse_clean["Description"].isna().sum()
)

display(
    df_eclipse_clean[
        ["Issue_id", "issue_text", "Duplicated_issue"]
    ].head()
)

Original shape: (85156, 11)
Cleaned shape: (85156, 12)
Empty issue texts: 1
Missing descriptions after cleaning: 0


,Issue_id,issue_text,Duplicated_issue
0,1,Usability issue with external editors (1GE6IRL...,NaN
1,2,Opening repository resources doesnt honor type...,NaN
2,3,Sync does not indicate deletion (1GIEN83)\nKM ...,NaN
3,4,need better error message if catching up over ...,NaN
4,5,ISharingManager sharing API inconsistent (1GAU...,NaN


In [8]:
duplicate_reports = df_eclipse_clean[
    df_eclipse_clean["Duplicated_issue"].notna()
].copy()

duplicate_reports["Duplicated_issue"] = (
    duplicate_reports["Duplicated_issue"].astype("int64")
)

target_reports = df_eclipse_clean[
    ["Issue_id", "issue_text", "Created_time"]
].rename(
    columns={
        "Issue_id": "target_issue_id",
        "issue_text": "target_text",
        "Created_time": "target_created_at"
    }
)

eclipse_pairs = duplicate_reports.merge(
    target_reports,
    left_on="Duplicated_issue",
    right_on="target_issue_id",
    how="inner",
    validate="many_to_one"
)

print("Valid duplicate pairs:", len(eclipse_pairs))

print(
    "Unique source issues:",
    eclipse_pairs["Issue_id"].nunique()
)

print(
    "Unique target issues:",
    eclipse_pairs["target_issue_id"].nunique()
)

display(
    eclipse_pairs[
        [
            "Issue_id",
            "target_issue_id",
            "issue_text",
            "target_text"
        ]
    ].head()
)

Valid duplicate pairs: 13231
Unique source issues: 13231
Unique target issues: 8101


,Issue_id,target_issue_id,issue_text,target_text
0,25,28126,[CVS UI] Need VCM Prefs - Default Repo Connect...,[Wizards] Patch: Standard public cvs repositor...
1,40,20,Need connect to team stream (1GCQPKW)\nI would...,Workspace files\nThought it would be useful if...
2,48,22,Make sure can future store other project refer...,persist sharing recommendations and project ve...
3,61,60,.vcm_meta showing as change (1GDQTGW)\nUser 1 ...,Need custom .vcm_ignore compare/merge (1GDQT96...
4,94,2,Repositories View: all file types open to the ...,Opening repository resources doesnt honor type...


In [9]:

eclipse_pairs["source_created_at"] = eclipse_pairs["Created_time"]

eclipse_pairs["target_created_before_source"] = (
    eclipse_pairs["target_created_at"]
    <= eclipse_pairs["source_created_at"]
)

print(
    "Target created before or at source:",
    eclipse_pairs["target_created_before_source"].sum()
)

print(
    "Target created after source:",
    (~eclipse_pairs["target_created_before_source"]).sum()
)

print("\nPercentage with earlier or same-time targets:")

print(
    round(
        eclipse_pairs["target_created_before_source"].mean() * 100,
        2
    ),
    "%"
)

Target created before or at source: 10387
Target created after source: 2844

Percentage with earlier or same-time targets:
78.51 %


In [10]:
source_ids = set(eclipse_pairs["Issue_id"])
target_ids = set(eclipse_pairs["target_issue_id"])

overlapping_ids = source_ids.intersection(target_ids)

print(
    "Issues appearing as both source and target:",
    len(overlapping_ids)
)

overlapping_pairs = eclipse_pairs[
    eclipse_pairs["Issue_id"].isin(overlapping_ids)
    | eclipse_pairs["target_issue_id"].isin(overlapping_ids)
]

display(
    overlapping_pairs[
        ["Issue_id", "target_issue_id"]
    ].head(15)
)

Issues appearing as both source and target: 633


,Issue_id,target_issue_id
11,178,22806
15,215,123
16,226,178
28,1867,8822
29,1878,2266
30,1947,2172
35,2041,2859
39,2090,37681
41,2110,7790
44,2147,2041


In [11]:
target_counts = (
    eclipse_pairs["target_issue_id"]
    .value_counts()
)

print("Unique targets:", len(target_counts))

print(
    "Targets referenced more than once:",
    (target_counts > 1).sum()
)

print(
    "Maximum duplicate references to one target:",
    target_counts.max()
)

print("\nTop 10 most referenced targets:")
display(target_counts.head(10))

Unique targets: 8101
Targets referenced more than once: 2432
Maximum duplicate references to one target: 38

Top 10 most referenced targets:


target_issue_id
214092    38
24448     31
47296     25
291257    24
56390     22
142965    21
78099     18
55537     17
128455    17
11624     16
Name: count, dtype: int64

In [12]:
duplicate_map = dict(
    zip(
        eclipse_pairs["Issue_id"],
        eclipse_pairs["target_issue_id"]
    )
)

cycles = set()
visited = set()

for start in duplicate_map:
    if start in visited:
        continue

    path = []
    positions = {}
    current = start

    while current in duplicate_map:
        if current in positions:
            cycle = path[positions[current]:]
            cycles.add(tuple(sorted(cycle)))
            break

        if current in visited:
            break

        positions[current] = len(path)
        path.append(current)
        current = duplicate_map[current]

    visited.update(path)

print("Number of detected cycles:", len(cycles))

print("Examples:")
for cycle in list(sorted(cycles))[:5]:
    print(cycle)

Number of detected cycles: 0
Examples:


In [13]:
output_dir = project_root / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

issues_path = output_dir / "eclipse_issues_clean.csv"
pairs_path = output_dir / "eclipse_duplicate_pairs.csv"

df_eclipse_clean.to_csv(
    issues_path,
    index=False
)

eclipse_pairs.to_csv(
    pairs_path,
    index=False
)

print("Saved:", issues_path)
print("Shape:", df_eclipse_clean.shape)

print("\nSaved:", pairs_path)
print("Shape:", eclipse_pairs.shape)

Saved: c:\Users\user\Documents\fisayo\duplicate-bug-report-detection\data\processed\eclipse_issues_clean.csv
Shape: (85156, 12)

Saved: c:\Users\user\Documents\fisayo\duplicate-bug-report-detection\data\processed\eclipse_duplicate_pairs.csv
Shape: (13231, 17)
